To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth your local device, follow [our guide](https://docs.unsloth.ai/get-started/install-and-update). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News


Unsloth's [Docker image](https://hub.docker.com/r/unsloth/unsloth) is here! Start training with no setup & environment issues. [Read our Guide](https://docs.unsloth.ai/new/how-to-train-llms-with-unsloth-and-docker).

[gpt-oss RL](https://docs.unsloth.ai/new/gpt-oss-reinforcement-learning) is now supported with the fastest inference & lowest VRAM. Try our [new notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/gpt-oss-(20B)-GRPO.ipynb) which creates kernels!

Introducing [Vision](https://docs.unsloth.ai/new/vision-reinforcement-learning-vlm-rl) and [Standby](https://docs.unsloth.ai/basics/memory-efficient-rl) for RL! Train Qwen, Gemma etc. VLMs with GSPO - even faster with less VRAM.

Unsloth now supports Text-to-Speech (TTS) models. Read our [guide here](https://docs.unsloth.ai/basics/text-to-speech-tts-fine-tuning).

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [ ]:
%%capture
!pip install weave
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
    except: get_numpy = "numpy"; get_pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {get_numpy} {get_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@05b2c186c1b6c9a08375389d5efe9cb4c401c075#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

### Unsloth

In [3]:
####################################
# IMPROVED TRAINING SCRIPT V2
# Poboljšanja:
# 1. ✅ System prompt dodat
# 2. ✅ Dataset preview (pre + tokom treninga)
# 3. ✅ Srpski metadata opciono
# 4. ✅ Fleksibilan izbor modela
# 5. ✅ Bolji logging i validacija
####################################

from unsloth import FastLanguageModel
import json
import torch
from datasets import Dataset
from transformers import TrainingArguments, TrainerCallback
from trl import SFTTrainer
import os
from datetime import datetime

# =================================================================================
# 1. MODEL SELECTION - IZABERI JEDAN! 🔽
# =================================================================================

# OPCIJA A: GPT-OSS (Harmony) - QLoRA (locked to Unsloth)
MODEL_CONFIG = {
    "name": "unsloth/gpt-oss-20b",
    "type": "gpt-oss",
    "load_in_4bit": True,
    "max_seq_length": 1024,
}

# OPCIJA B: GPT-OSS - LoRA + BF16 (exportable, ali >60GB VRAM!)
# MODEL_CONFIG = {
#     "name": "unsloth/gpt-oss-20b-BF16",
#     "type": "gpt-oss",
#     "load_in_4bit": False,
#     "max_seq_length": 1024,
# }

# OPCIJA C: Llama-3.1-8B (Standard, exportable, QLoRA)
# MODEL_CONFIG = {
#     "name": "unsloth/Meta-Llama-3.1-8B-Instruct",
#     "type": "llama-3",
#     "load_in_4bit": True,
#     "max_seq_length": 2048,
# }

# =================================================================================
# 2. SYSTEM PROMPT - KRITIČNO! 🔽
# =================================================================================

# System prompt za legal agent (Serbian CoT)
SYSTEM_PROMPT = """Ti si ekspertni pravni asistent specijalizovan za srpsko pravo. Tvoj zadatak je da odgovaraš na pravna pitanja uz jasno obrazloženje.

Kada odgovaraš:
1. Prvo analiziraj pitanje korak po korak
2. Razmotri relevantne zakone i članove
3. Objasni pravnu logiku
4. Daj jasan zaključak sa pravnim osnovom

Jezik razmišljanja: Srpski
Rezonovanje: Visoko
Stil: Precizan, zakonski utemeljen, razumljiv"""

# =================================================================================
# 3. DATASET & TRAINING CONFIGURATION
# =================================================================================

DATASET_PATH = "/content/harmony_300.jsonl"  # Promeni ako je drugde
OUTPUT_DIR = "lora_adapters"

# Training parameters
TRAINING_CONFIG = {
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "num_train_epochs": 2,
    "learning_rate": 2e-5,
    "logging_steps": 1,
    "optim": "adamw_8bit",
    "weight_decay": 0.01,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.08,
    "seed": 3407,
}

# LoRA parameters
LORA_CONFIG = {
    "r": 16,
    "alpha": 32,
    "dropout": 0,
    "bias": "none",
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
}

# Preview settings
PREVIEW_ENABLED = True
PREVIEW_INTERVAL = 100  # Prikaži svaki 100-ti training step

# =================================================================================
# 4. FORMATTING FUNCTIONS
# =================================================================================

def format_thinking_steps(thinking_list):
    """
    Formatira thinking steps u Chain-of-Thought format.
    Input: [{"channel": "analysis", "content": "..."}, ...] ili string
    Output: strukturisan CoT tekst
    """
    # Handle različite formate thinking podataka
    if isinstance(thinking_list, str):
        # Ako je već string, vrati ga
        return thinking_list

    if not isinstance(thinking_list, list):
        # Ako nije ni lista ni string, konvertuj u string
        return str(thinking_list)

    formatted_steps = []
    for step in thinking_list:
        # Ako je step dict sa "channel" i "content"
        if isinstance(step, dict):
            channel = step.get("channel", "analysis")
            content = step.get("content", "")
            formatted_steps.append(f"[{channel}] {content}")
        # Ako je step string
        elif isinstance(step, str):
            formatted_steps.append(f"[analysis] {step}")
        # Fallback za ostale tipove
        else:
            formatted_steps.append(f"[analysis] {str(step)}")

    return "\n".join(formatted_steps)


def format_training_example(original_json, model_type, add_system_prompt=True):
    """
    Formatira jedan primer iz dataseta u finalni training format.

    Args:
        original_json: Dict sa 'messages' ključem
        model_type: "gpt-oss" ili "llama-3"
        add_system_prompt: Da li dodati system prompt (DEFAULT: True!)

    Returns:
        Dict sa "text" ključem spremnim za SFTTrainer
    """
    try:
        messages = original_json.get('messages', [])
        if len(messages) < 2:
            raise ValueError("Nedovoljno poruka u primeru")

        user_message = messages[0]
        assistant_message = messages[1]

        # Extract content with safe fallbacks
        user_content = user_message.get('content', '')
        thinking_steps = assistant_message.get('thinking', [])

        # Handle različite strukture za final_answer
        assistant_content = assistant_message.get('content', {})
        if isinstance(assistant_content, dict):
            final_answer = assistant_content.get('text', '')
        elif isinstance(assistant_content, str):
            final_answer = assistant_content
        else:
            final_answer = str(assistant_content)

        # Format thinking steps (sada sa robust handlingom)
        thinking_text = format_thinking_steps(thinking_steps)

    except Exception as e:
        # Debug info ako nešto pukne
        print(f"⚠️  Greška u formatiranju primera: {e}")
        print(f"    Original JSON keys: {original_json.keys() if isinstance(original_json, dict) else 'Not a dict'}")
        if isinstance(original_json, dict) and 'messages' in original_json:
            print(f"    Messages length: {len(original_json['messages'])}")
            if len(original_json['messages']) > 1:
                print(f"    Assistant message keys: {original_json['messages'][1].keys()}")
        raise

    # Build final formatted text based on model type
    if model_type == "gpt-oss":
        # GPT-OSS (Harmony) format
        if add_system_prompt:
            formatted_text = f"""<|start|>system<|message|>
{SYSTEM_PROMPT}<|end|>
<|start|>user<|message|>
{user_content}<|end|>
<|start|>assistant<|message|>
<|thinking|>
{thinking_text}
<|end_thinking|>
{final_answer}<|end|>"""
        else:
            formatted_text = f"""<|start|>user<|message|>
{user_content}<|end|>
<|start|>assistant<|message|>
<|thinking|>
{thinking_text}
<|end_thinking|>
{final_answer}<|end|>"""

    elif model_type == "llama-3":
        # Llama-3 Instruct format
        if add_system_prompt:
            formatted_text = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{SYSTEM_PROMPT}<|eot_id|><|start_header_id|>user<|end_header_id|>

{user_content}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

<|thinking|>
{thinking_text}
<|end_thinking|>
{final_answer}<|eot_id|>"""
        else:
            formatted_text = f"""<|begin_of_text|><|start_header_id|>user<|end_header_id|>

{user_content}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

<|thinking|>
{thinking_text}
<|end_thinking|>
{final_answer}<|eot_id|>"""

    else:
        raise ValueError(f"Nepodržan model_type: {model_type}")

    return {"text": formatted_text}


def load_and_format_data(file_path, model_type, add_system_prompt=True):
    """
    Učitava dataset i formatira sve primere.
    """
    formatted_data = []
    skipped = 0

    print(f"\n{'='*80}")
    print(f"📚 UČITAVANJE DATASETA")
    print(f"{'='*80}")
    print(f"Putanja: {file_path}")
    print(f"Model tip: {model_type}")
    print(f"System prompt: {'✅ DA' if add_system_prompt else '❌ NE'}")
    print(f"{'='*80}\n")

    with open(file_path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            try:
                original_json = json.loads(line)
                formatted_example = format_training_example(
                    original_json,
                    model_type,
                    add_system_prompt
                )
                formatted_data.append(formatted_example)
            except (json.JSONDecodeError, ValueError, IndexError) as e:
                print(f"⚠️  Preskačem liniju {line_num}: {e}")
                skipped += 1

    print(f"\n✅ Dataset učitan:")
    print(f"   - Uspešno: {len(formatted_data)} primera")
    print(f"   - Preskočeno: {skipped} primera")
    print(f"{'='*80}\n")

    return formatted_data


def preview_dataset(dataset, num_examples=3):
    """
    Prikazuje primere iz dataseta pre treninga.
    """
    print(f"\n{'='*80}")
    print(f"🔍 PREVIEW DATASETA ({num_examples} primera)")
    print(f"{'='*80}\n")

    for i in range(min(num_examples, len(dataset))):
        print(f"📄 PRIMER {i+1}/{min(num_examples, len(dataset))}")
        print(f"{'-'*80}")
        print(dataset[i]['text'])
        print(f"{'-'*80}\n")

        # Provera dužine
        text_length = len(dataset[i]['text'])
        print(f"📏 Dužina: {text_length} karaktera\n")

        if i < min(num_examples, len(dataset)) - 1:
            print(f"\n{'~'*80}\n")

    print(f"{'='*80}\n")


# =================================================================================
# 5. TRAINING CALLBACK - Za preview tokom treninga
# =================================================================================

class PreviewCallback(TrainerCallback):
    """
    Callback koji prikazuje primere tokom treninga.
    """
    def __init__(self, dataset, interval=100):
        self.dataset = dataset
        self.interval = interval
        self.step_count = 0

    def on_log(self, args, state, control, logs=None, **kwargs):
        """Poziva se na svakom logging koraku."""
        self.step_count += 1

        if self.step_count % self.interval == 0:
            print(f"\n{'='*80}")
            print(f"📊 TRAINING STEP {self.step_count}")
            print(f"{'='*80}")

            # Prikaži random primer iz dataseta
            import random
            random_idx = random.randint(0, len(self.dataset) - 1)
            example = self.dataset[random_idx]

            print(f"\n📄 Random primer iz dataseta (#{random_idx}):")
            print(f"{'-'*80}")
            print(example['text'][:500] + "..." if len(example['text']) > 500 else example['text'])
            print(f"{'-'*80}")

            # Prikaži metrics ako postoje
            if logs:
                print(f"\n📈 Metrics:")
                for key, value in logs.items():
                    print(f"   {key}: {value}")

            print(f"\n{'='*80}\n")


# =================================================================================
# 6. MAIN TRAINING PIPELINE
# =================================================================================

def main():
    print(f"\n{'#'*80}")
    print(f"# IMPROVED TRAINING SCRIPT V2")
    print(f"# Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'#'*80}\n")

    # Check dataset exists
    if not os.path.exists(DATASET_PATH):
        raise FileNotFoundError(
            f"Dataset nije pronađen na '{DATASET_PATH}'. "
            f"Uploaduj ga ili promeni DATASET_PATH u konfiguraciji."
        )

    # Load and format dataset
    formatted_list = load_and_format_data(
        DATASET_PATH,
        MODEL_CONFIG["type"],
        add_system_prompt=True  # 🔥 KRITIČNO: System prompt uključen!
    )

    dataset = Dataset.from_list(formatted_list)
    print(f"✅ HuggingFace Dataset kreiran: {len(dataset)} primera\n")

    # Preview dataset pre treninga
    if PREVIEW_ENABLED:
        preview_dataset(dataset, num_examples=3)

    # Save dataset to disk
    dataset.save_to_disk('chovus_hf_dataset')
    print(f"💾 Dataset sačuvan na disk: 'chovus_hf_dataset'\n")

    # Load model
    print(f"{'='*80}")
    print(f"🤖 UČITAVANJE MODELA")
    print(f"{'='*80}")
    print(f"Model: {MODEL_CONFIG['name']}")
    print(f"4-bit: {MODEL_CONFIG['load_in_4bit']}")
    print(f"Max seq length: {MODEL_CONFIG['max_seq_length']}")
    print(f"{'='*80}\n")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_CONFIG["name"],
        max_seq_length=MODEL_CONFIG["max_seq_length"],
        dtype=None,  # Auto-detect
        load_in_4bit=MODEL_CONFIG["load_in_4bit"],
    )

    # Add LoRA adapters
    print(f"{'='*80}")
    print(f"🔧 DODAVANJE LoRA ADAPTERA")
    print(f"{'='*80}")
    print(f"Rank (r): {LORA_CONFIG['r']}")
    print(f"Alpha: {LORA_CONFIG['alpha']}")
    print(f"Target modules: {', '.join(LORA_CONFIG['target_modules'])}")
    print(f"{'='*80}\n")

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_CONFIG["r"],
        target_modules=LORA_CONFIG["target_modules"],
        lora_alpha=LORA_CONFIG["alpha"],
        lora_dropout=LORA_CONFIG["dropout"],
        bias=LORA_CONFIG["bias"],
        use_gradient_checkpointing="unsloth",
        random_state=TRAINING_CONFIG["seed"],
        max_seq_length=MODEL_CONFIG["max_seq_length"],
    )

    # Setup trainer
    print(f"{'='*80}")
    print(f"⚙️  TRAINING KONFIGURACIJA")
    print(f"{'='*80}")
    for key, value in TRAINING_CONFIG.items():
        print(f"   {key}: {value}")
    print(f"{'='*80}\n")

    # Create callback
    preview_callback = PreviewCallback(dataset, interval=PREVIEW_INTERVAL) if PREVIEW_ENABLED else None

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=MODEL_CONFIG["max_seq_length"],
        packing=False,
        args=TrainingArguments(
            per_device_train_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
            gradient_accumulation_steps=TRAINING_CONFIG["gradient_accumulation_steps"],
            warmup_ratio=TRAINING_CONFIG["warmup_ratio"],
            num_train_epochs=TRAINING_CONFIG["num_train_epochs"],
            learning_rate=TRAINING_CONFIG["learning_rate"],
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=TRAINING_CONFIG["logging_steps"],
            optim=TRAINING_CONFIG["optim"],
            weight_decay=TRAINING_CONFIG["weight_decay"],
            lr_scheduler_type=TRAINING_CONFIG["lr_scheduler_type"],
            seed=TRAINING_CONFIG["seed"],
            output_dir=OUTPUT_DIR,
        ),
        callbacks=[preview_callback] if preview_callback else [],
    )

    # Start training
    print(f"\n{'#'*80}")
    print(f"# 🚀 ZAPOČINJEM TRAINING")
    print(f"{'#'*80}\n")

    trainer.train()

    print(f"\n{'#'*80}")
    print(f"# ✅ TRAINING ZAVRŠEN!")
    print(f"# End: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'#'*80}\n")

    # Save model
    print(f"{'='*80}")
    print(f"💾 ČUVANJE MODELA")
    print(f"{'='*80}")

    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)

    print(f"✅ Model i tokenizer sačuvani u: '{OUTPUT_DIR}'")
    print(f"{'='*80}\n")

    print(f"\n{'#'*80}")
    print(f"# 🎉 SVE GOTOVO!")
    print(f"# Možeš downloadovati folder: '{OUTPUT_DIR}'")
    print(f"{'#'*80}\n")


if __name__ == "__main__":
    main()



################################################################################
# IMPROVED TRAINING SCRIPT V2
# Start: 2025-10-31 12:00:56
################################################################################


📚 UČITAVANJE DATASETA
Putanja: /content/harmony_300.jsonl
Model tip: gpt-oss
System prompt: ✅ DA


✅ Dataset učitan:
   - Uspešno: 992 primera
   - Preskočeno: 0 primera

✅ HuggingFace Dataset kreiran: 992 primera


🔍 PREVIEW DATASETA (3 primera)

📄 PRIMER 1/3
--------------------------------------------------------------------------------
<|start|>system<|message|>
Ti si ekspertni pravni asistent specijalizovan za srpsko pravo. Tvoj zadatak je da odgovaraš na pravna pitanja uz jasno obrazloženje.

Kada odgovaraš:
1. Prvo analiziraj pitanje korak po korak
2. Razmotri relevantne zakone i članove
3. Objasni pravnu logiku
4. Daj jasan zaključak sa pravnim osnovom

Jezik razmišljanja: Srpski
Rezonovanje: Visoko
Stil: Precizan, zakonski utemeljen, razumljiv<|end|>
<|star

Saving the dataset (0/1 shards):   0%|          | 0/992 [00:00<?, ? examples/s]

💾 Dataset sačuvan na disk: 'chovus_hf_dataset'

🤖 UČITAVANJE MODELA
Model: unsloth/gpt-oss-20b
4-bit: True
Max seq length: 1024

==((====))==  Unsloth 2025.10.12: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gpt_oss won't work! Using float32.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

...

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: chovus (chovus-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Weave is installed but not imported. Add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,9.246200
2,9.377200
3,9.159700
4,9.258500
5,8.797300
6,8.723100
7,7.343000
8,7.351500
9,7.036000
10,5.903900



📊 TRAINING STEP 100

📄 Random primer iz dataseta (#58):
--------------------------------------------------------------------------------
<|start|>system<|message|>
Ti si ekspertni pravni asistent specijalizovan za srpsko pravo. Tvoj zadatak je da odgovaraš na pravna pitanja uz jasno obrazloženje.

Kada odgovaraš:
1. Prvo analiziraj pitanje korak po korak
2. Razmotri relevantne zakone i članove
3. Objasni pravnu logiku
4. Daj jasan zaključak sa pravnim osnovom

Jezik razmišljanja: Srpski
Rezonovanje: Visoko
Stil: Precizan, zakonski utemeljen, razumljiv<|end|>
<|start|>user<|message|>
Policija me je uhvatila sa 2 grama marihuane, k...
--------------------------------------------------------------------------------

📈 Metrics:
   loss: 1.7561
   grad_norm: 0.5508228540420532
   learning_rate: 2.2808334908367914e-06
   epoch: 1.6129032258064515



################################################################################
# ✅ TRAINING ZAVRŠEN!
# End: 2025-10-31 15:01:21
#############

In [4]:
import os
from google.colab import files

# Define the directory to zip
output_dir = "lora_adapters"

# Create a zip archive of the directory
zip_filename = f"{output_dir}.zip"
!zip -r {zip_filename} {output_dir}

# Provide a link to download the zip file
files.download(zip_filename)

  adding: lora_adapters/ (stored 0%)
  adding: lora_adapters/runs/ (stored 0%)
  adding: lora_adapters/runs/Oct31_12-04-08_cb21ba10d769/ (stored 0%)
  adding: lora_adapters/runs/Oct31_12-04-08_cb21ba10d769/events.out.tfevents.1761912263.cb21ba10d769.832.0 (deflated 68%)
  adding: lora_adapters/chat_template.jinja (deflated 75%)
  adding: lora_adapters/adapter_model.safetensors (deflated 8%)
  adding: lora_adapters/tokenizer_config.json (deflated 89%)
  adding: lora_adapters/adapter_config.json (deflated 57%)
  adding: lora_adapters/checkpoint-124/ (stored 0%)
  adding: lora_adapters/checkpoint-124/chat_template.jinja (deflated 75%)
  adding: lora_adapters/checkpoint-124/adapter_model.safetensors (deflated 8%)
  adding: lora_adapters/checkpoint-124/tokenizer_config.json (deflated 89%)
  adding: lora_adapters/checkpoint-124/training_args.bin (deflated 53%)
  adding: lora_adapters/checkpoint-124/adapter_config.json (deflated 57%)
  adding: lora_adapters/checkpoint-124/optimizer.pt (deflat

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

We're about to demonstrate the power of the new OpenAI GPT-OSS 20B model through a finetuning example. To use our `MXFP4` inference example, use this [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/GPT_OSS_MXFP4_(20B)-Inference.ipynb) instead.

In [ ]:
# final_training_script.py

# =================================================================================
# 1. SETUP
# =================================================================================
#
# In your Google Colab notebook, create a new code cell and paste the following commands
# to install the necessary libraries. Run this cell first.
#
# !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# !pip install --no-deps "xformers<0.0.26" "trl<0.9.0" "peft<0.11.0" "accelerate<0.30.0" "bitsandbytes<0.44.0"
#
# After installation, you must restart the Colab runtime.
# (Runtime -> Restart session)
#
# Also, make sure to upload your 'harmony_prepared.jsonl' file to the Colab environment,
# or mount your Google Drive and provide the correct path in the configuration below.
#
# =================================================================================

from unsloth import FastLanguageModel
import json
import torch
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer
import os
import weave

# =================================================================================
# 2. CONFIGURATION
# =================================================================================

# --- Dataset Configuration ---
# Path to the dataset you uploaded to Colab
DATASET_PATH = "/content/harmony_prepared.jsonl"

# --- Model Configuration ---
# Llama-3 8B is set as the default because it is guaranteed to work on a 10-12GB GPU.
model_name = "unsloth/gpt-oss-20b"

# !! WARNING !!
# gpt-oss-20b requires > 12GB of VRAM, even with 4-bit quantization.
# It will likely cause a CUDA Out of Memory error on a 10GB GPU.
# If you have a larger GPU (e.g., 16GB+), you can try uncommenting the line below.
# model_name = "unsloth/gpt-oss-20b-chat-fp16"

# --- VRAM/Memory Optimization ---
# If you get a CUDA Out of Memory error, the first thing to do is to lower max_seq_length!
max_seq_length = 1024 # Default: 2048. Try 1024 or 512 if you run out of memory.
dtype = None # None for auto detection. Float16 for Tesla T4/V100, Bfloat16 for Ampere.
load_in_4bit = True

# --- LoRA Configuration ---
# If you get OOM, you can also try lowering lora_r to 8.
lora_r = 8
lora_alpha = 16
lora_dropout = 0
lora_bias = "none"
lora_target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# --- Training Configuration ---
output_dir = "lora_adapters" # Directory to save the trained LoRA adapters
per_device_train_batch_size = 1
gradient_accumulation_steps = 8
warmup_steps = 25
#max_steps = 30
num_train_epochs = 2
learning_rate = 2e-5
fp16 = not torch.cuda.is_bf16_supported()
bf16 = torch.cuda.is_bf16_supported()
logging_steps = 1
optim = "adamw_8bit"
weight_decay = 0.01
lr_scheduler_type = "linear"
use_gradient_checkpointing = "unsloth" # True or "unsloth" for very long context
seed = 3407
use_rslora = False  # We support rank stabilized LoRA
loftq_config = None # And LoftQ

# =================================================================================
# 3. DATA PREPARATION
# =================================================================================

def load_and_format_data(file_path, model_name_for_template):
    """
    Loads data from a JSONL file, applies the correct model-specific prompt formatting,
    and returns a clean list of dictionaries ready for Dataset creation.
    """
    formatted_data = []
    print("Starting manual data loading and formatting...")

    # Check which template to use
    is_llama3_model = "llama-3" in model_name_for_template.lower()
    if not is_llama3_model:
        print("Using Harmony (gpt-oss) chat template.")
    else:
        print("Using Llama-3 Instruct chat template.")

    with open(file_path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            try:
                original_json = json.loads(line)

                messages = original_json.get('messages', [])
                if len(messages) < 2:
                    print(f"Warning: Skipping line {line_num} due to not having enough messages.")
                    continue

                user_message = messages[0]
                assistant_message = messages[1]

                thinking_steps_json = json.dumps(assistant_message.get('thinking', []), ensure_ascii=False, indent=2)
                final_answer = assistant_message.get('content', {}).get('text', '')

                # Apply the correct template based on the model name
                if is_llama3_model:
                    # Llama-3 Instruct format
                    formatted_text = f"""<|start_header_id|>user<|end_header_id|>

{user_message['content']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

<|thinking|>
{thinking_steps_json}
<|end_thinking|>
{final_answer}<|eot_id|>"""
                else:
                    # Harmony (gpt-oss) format
                    formatted_text = f"""<|start|>user<|message|>
{user_message['content']}<|end|>
<|start|>assistant<|message|>
<|thinking|>
{thinking_steps_json}
<|end_thinking|>
{final_answer}<|end|>"""

                formatted_data.append({"text": formatted_text})

            except (json.JSONDecodeError, IndexError) as e:
                print(f"Warning: Skipping line {line_num} due to error: {e}")

    print(f"Manual loading and formatting complete. Total examples: {len(formatted_data)}")
    return formatted_data

# Check if the dataset file exists
if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f"Dataset file not found at '{DATASET_PATH}'. Please upload it to your Colab environment.")

# 1. Load data and format it in pure Python to create a simple, consistent list
# Pass the model_name to the formatting function to apply the correct template
formatted_list = load_and_format_data(DATASET_PATH, model_name)

# 2. Create the HuggingFace Dataset from the clean list
dataset = Dataset.from_list(formatted_list)

print("HuggingFace Dataset created successfully.")
if len(dataset) > 0:
    print("\nAn example of a formatted prompt:")
    print("="*80)
    print(dataset[0]['text'])
    print("="*80)


# =================================================================================
# 4. MODEL TRAINING
# =================================================================================

print("Loading base model...")
# Load the model using Unsloth's FastLanguageModel for memory and speed optimizations
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("Patching model with PEFT (LoRA)...")
# Add LoRA adapters to the model to enable efficient fine-tuning
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_r,
    target_modules = lora_target_modules,
    lora_alpha = lora_alpha,
    lora_dropout = lora_dropout,
    bias = lora_bias,
    use_gradient_checkpointing = True,
    random_state = seed,
    max_seq_length = max_seq_length,
)

print("Setting up the SFTTrainer...")
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    # dataset_num_proc = 1, # Number of processes to use for data preparation
    packing = False, # Can be set to True for faster training, but requires more VRAM
    args = TrainingArguments(
        per_device_train_batch_size = per_device_train_batch_size,
        gradient_accumulation_steps = gradient_accumulation_steps,
        warmup_steps = warmup_steps,
        num_train_epochs = num_train_epochs,
        learning_rate = learning_rate,
        fp16 = fp16,
        bf16 = bf16,
        logging_steps = logging_steps,
        optim = optim,
        weight_decay = weight_decay,
        lr_scheduler_type = lr_scheduler_type,
        seed = seed,
        output_dir = output_dir,
    ),
)

print("""
=================================================================================
--- Starting Training ---
=================================================================================
""")
trainer.train()
print("--- Training Finished ---")


# =================================================================================
# 5. SAVE THE MODEL
# =================================================================================

print(f"Saving LoRA adapters to '{output_dir}'...")
model.save_pretrained(output_dir)
# Also save the tokenizer
tokenizer.save_pretrained(output_dir)
print("Adapters and tokenizer saved successfully!")
print(f"\nYou can download the '{output_dir}' folder from your Colab environment.")


In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 1024
dtype = None

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/gpt-oss-20b-unsloth-bnb-4bit", # 20B model using bitsandbytes 4bit quantization
    "unsloth/gpt-oss-120b-unsloth-bnb-4bit",
    "unsloth/gpt-oss-20b", # 20B model using MXFP4 format
    "unsloth/gpt-oss-120b",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b",
    dtype = dtype, # None for auto detection
    max_seq_length = max_seq_length, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

### Reasoning Effort
The `gpt-oss` models from OpenAI include a feature that allows users to adjust the model's "reasoning effort." This gives you control over the trade-off between the model's performance and its response speed (latency) which by the amount of token the model will use to think.

----

The `gpt-oss` models offer three distinct levels of reasoning effort you can choose from:

* **Low**: Optimized for tasks that need very fast responses and don't require complex, multi-step reasoning.
* **Medium**: A balance between performance and speed.
* **High**: Provides the strongest reasoning performance for tasks that require it, though this results in higher latency.

In [ ]:
from transformers import TextStreamer

messages = [
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "low", # **NEW!** Set reasoning effort to low, medium or high
).to("cuda")

_ = model.generate(**inputs, max_new_tokens = 64, streamer = TextStreamer(tokenizer))

Changing the `reasoning_effort` to `medium` will make the model think longer. We have to increase the `max_new_tokens` to occupy the amount of the generated tokens but it will give better and more correct answer

In [ ]:
from transformers import TextStreamer

messages = [
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "medium", # **NEW!** Set reasoning effort to low, medium or high
).to("cuda")

_ = model.generate(**inputs, max_new_tokens = 64, streamer = TextStreamer(tokenizer))

Lastly we will test it using `reasoning_effort` to `high`

In [ ]:
from transformers import TextStreamer

messages = [
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "high", # **NEW!** Set reasoning effort to low, medium or high
).to("cuda")

_ = model.generate(**inputs, max_new_tokens = 64, streamer = TextStreamer(tokenizer))

<a name="Data"></a>
### Data Prep

The `HuggingFaceH4/Multilingual-Thinking` dataset will be utilized as our example. This dataset, available on Hugging Face, contains reasoning chain-of-thought examples derived from user questions that have been translated from English into four other languages. It is also the same dataset referenced in OpenAI's [cookbook](https://cookbook.openai.com/articles/gpt-oss/fine-tune-transfomers) for fine-tuning. The purpose of using this dataset is to enable the model to learn and develop reasoning capabilities in these four distinct languages.

In [ ]:
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

from datasets import load_dataset

dataset = load_dataset("HuggingFaceH4/Multilingual-Thinking", split="train")
dataset

To format our dataset, we will apply our version of the GPT OSS prompt

In [ ]:
from unsloth.chat_templates import standardize_sharegpt
dataset = standardize_sharegpt(dataset)
dataset = dataset.map(formatting_prompts_func, batched = True,)

Let's take a look at the dataset, and check what the 1st example shows

In [ ]:
print(dataset[0]['text'])

What is unique about GPT-OSS is that it uses OpenAI [Harmony](https://github.com/openai/harmony) format which support conversation structures, reasoning output, and tool calling.

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [ ]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 30,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes and lower loss as well!

In [ ]:
from unsloth.chat_templates import train_on_responses_only

gpt_oss_kwargs = dict(instruction_part = "<|start|>user<|message|>", response_part="<|start|>assistant<|channel|>final<|message|>")

trainer = train_on_responses_only(
    trainer,
    **gpt_oss_kwargs,
)

Let's verify masking the instruction part is done! Let's print the 100th row again.

In [ ]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

Now let's print the masked out example - you should see only the answer is present:

In [ ]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!

In [ ]:
messages = [
    {"role": "system", "content": "reasoning language: French\n\nYou are a helpful assistant that can solve mathematical problems."},
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "medium",
).to("cuda")
from transformers import TextStreamer
_ = model.generate(**inputs, max_new_tokens = 64, streamer = TextStreamer(tokenizer))

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** Currently finetunes can only be loaded via Unsloth in the meantime - we're working on vLLM and GGUF exporting!

In [ ]:
model.save_pretrained("finetuned_model")
# model.push_to_hub("hf_username/finetuned_model", token = "hf_...") # Save to HF

To run the finetuned model, you can do the below after setting `if False` to `if True` in a new instance.

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "finetuned_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 1024,
        dtype = None,
        load_in_4bit = True,
    )

messages = [
    {"role": "system", "content": "reasoning language: French\n\nYou are a helpful assistant that can solve mathematical problems."},
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "high",
).to("cuda")
from transformers import TextStreamer
_ = model.generate(**inputs, max_new_tokens = 64, streamer = TextStreamer(tokenizer))

### Saving to float16 for VLLM or mxfp4

We also support saving to `float16` or `mxfp4` directly. Select `merged_16bit` for float16. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge and push to hub in mxfp4 4bit format
if False:
    model.save_pretrained_merged("finetuned_model", tokenizer, save_method = "mxfp4")
if False: model.push_to_hub_merged("repo_id/repo_name", tokenizer, token = "hf...", save_method = "mxfp4")

# Merge and push to hub in 16bit
if False:
    model.save_pretrained_merged("finetuned_model", tokenizer, save_method = "merged_16bit")
if False: # Pushing to HF Hub
    model.push_to_hub_merged("hf/gpt-oss-finetune", tokenizer, save_method = "merged_16bit", token = "")

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).
